# Формат ответов MCP

Почему в content всегда плоский текст и как в нём выглядят секции, статьи, темы, списки, записи и ошибки. Формат общий для /mcp/kb и /mcp/kb-admin.

## Плоский текст в content

Правило для всего MCP: `/mcp/kb` (все профили) и `/mcp/kb-admin`. Настройки формата нет.

- В `content` ответа — **всегда** плоский текст: тексты секций без JSON-экранирования, списки — по строке на элемент. Причина: модель читает `content` напрямую; JSON там — лишние токены и экранирование, которое слабые модели потом переносят в запись.
- Тот же смысл — JSON в `structuredContent` (`{"ok": true, "result": …}` / `{"ok": false, "error": {…}}`) — для программ и тестов. Рекомендацию MCP дублировать JSON в `content` сознательно не выполняем (это SHOULD, не MUST).
- `outputSchema` у инструментов **не объявляем**: при ней клиенты обязаны проверять `structuredContent` и могут начать показывать модели его, а не `content`.
- Клиентам, которые всё равно показывают модели `structuredContent` (Claude Code), — `off=structured`, см. `development/mcp-kb/api#connection`.
- Ошибка — дополнительно флаг MCP `isError: true`; текст начинается с `ERROR <code>:`.
- Пустой результат — не пустая строка, а явная `(no results)` / `(нет результатов)` по `lang`, плюс подсказка (укоротить подстроку и т.п.).
- Ключи строк (`title:`, `tags:`, `Hint:` …), коды и пути — латиница; служебные фразы — по `lang`.

**Входные параметры** приходят на сервер JSON-объектом (так устроен MCP). В каком виде модель пишет вызов, определяют её формат вызова и парсер рантайма — это вне нашего контроля. Схема держится плоской и короткой; теги принимаются и списком, и строкой через запятую.

## Секции, статьи, темы и списки

**Блок секции** — одинаковый везде, где отдаётся секция (`get` секции, полная статья, `sections_by_tag`). Начинается строкой `=== <путь>`, заканчивается перед следующей `=== ` или концом ответа:
```
=== coding/python/asyncio#event-loop
title: Event loop
tags: event loop, concurrency
---
Текст секции как есть...
```
`tags:` — только при `with_tags`. Всё после `---` — тело как есть (в нём могут быть свои `---`).

**Статья** (`get` по умолчанию, оглавление): блок шапки целиком, затем остальные секции строками списка:
```
=== coding/python/asyncio#head
title: Asyncio
---
Текст шапки...

Sections:
coding/python/asyncio#event-loop — Event loop
coding/python/asyncio#tasks — Задачи
```
С `full=true` — подряд блоки всех секций в порядке файла.

**Тема**: блок шапки темы, затем `Topics:` и `Articles:` — списки «путь — название». **Корень** `get("")` — только список областей.

**Список** (`index`, `search*`, `section_tags`, `search_section_tags*`):
```
coding/python/asyncio — Asyncio
coding/python/packaging/wheels — Wheels
```
Теги — по тегу на строку, без путей.

## Записи и ошибки

**Запись:**
```
OK created coding/python/asyncio#event-loop
OK updated coding/python/asyncio#event-loop
OK unchanged coding/python/asyncio#event-loop
Note: path normalized to coding/python/asyncio
```
`Note:` — отдельной строкой в любом ответе, где путь нормализован.

**Ошибка:**
```
ERROR not_found: Section 'coding/python/asyncio#evnt-loop' not found.
Hint: pick a path below or call get("coding/python/asyncio").
Options:
- coding/python/asyncio#event-loop — Event loop
- coding/python/asyncio#tasks — Задачи
```

## Поля секции: title, content, tags

```json
{"path": "coding/python/asyncio#event-loop", "title": "Event loop", "content": "…", "tags": ["event loop", "concurrency"]}
```
- `title` — заголовок без `#`/`##`; `content` — текст без строки заголовка.
- Строку заголовка ставит и снимает сервер: заголовок — **только первая строка** ячейки. Всё остальное — тело как есть; строки `#`/`##` внутри тела секции секцию не делят (в оглавлении Jupyter они видны как заголовки — это поправит индексатор).
- `tags` — только при `with_tags`; только тексты, без источника (источник видно в админке).